# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This notebook establishes the research question, decision framing, data contract evidence, and careful claim boundaries for Week 1.

## 1. My lane (or freestyle) and why

**Provisional Lane Selected**: **Lane 2 — Refresh / Content Opportunity Scoring**

**Why this lane?**
Content marketing and SEO teams managing large multi-client portfolios face severe capacity constraints: thousands of published pages exist across clients, but editorial teams can only review and refresh a handful of pages per week (e.g., 20–50 pages). Relying on simple manual rules (like sorting by age or traffic) either flags too many healthy high-traffic pages or misses stale pages undergoing silent traffic decay. Lane 2 focuses on building a decision-support system that ranks content by refresh opportunity, combining visibility, age, position, CTR, and trend signals so human editors spend their limited review time on the highest-impact candidates first.

In [1]:
# 1. Lane Selection Summary Verification
selected_lane = 'Lane 2: Refresh / Content Opportunity Scoring'
print(f'Provisional Lane Confirmed: {selected_lane}')
print('Focus: Ranking content for editorial refresh review prioritization.')


Provisional Lane Confirmed: Lane 2: Refresh / Content Opportunity Scoring
Focus: Ranking content for editorial refresh review prioritization.


## 2. The question: decision, action, cost of a wrong call

**The Four Framing Questions:**

1. **What decision does this improve?**
   - It answers: *"Which existing content items across our client portfolio should an editorial team prioritize for content refresh review this week?"*

2. **Who acts on the output, and what do they do?**
   - **User**: Content editors, SEO managers, and copywriters.
   - **Action**: Perform an editorial audit on flagged pages—updating outdated stats, refreshing search intent alignment, fixing broken/outdated links, or improving title/meta snippets.

3. **What does a wrong recommendation cost?**
   - *False Positive (Flagging a healthy/stable page)*: Costs wasted editorial hours auditing a page that didn't need updates (opportunity cost of missing a truly declining page).
   - *False Negative (Missing a decaying high-traffic page)*: Costs cumulative organic search traffic and revenue as the page continues to lose search visibility silently.

4. **Why does data / ML help at all?**
   - Single-variable rules (e.g. `days_since_last_update >= 180`) are rigid and noisy: many old pages maintain steady rankings while younger pages decay rapidly due to search intent shifts. Machine learning integrates non-linear interactions across 40+ signals (impressions, position tiers, CTR, engagement, scroll rate, content age, and trend metrics) to produce a dynamic, evidence-backed priority queue.

In [2]:
# 2. Decision Framing Verification Matrix
framing_matrix = {
    'Target Audience': 'SEO Managers & Content Editors',
    'Decision': 'Weekly content refresh priority ranking',
    'Action': 'Editorial audit, content expansion, meta/title updates',
    'Evaluation Metric': 'Precision@50 & Top-K Review Efficiency',
    'Primary Risk': 'Wasted editorial audit hours on false positives'
}
for key, val in framing_matrix.items():
    print(f'{key:22s}: {val}')


Target Audience       : SEO Managers & Content Editors
Decision              : Weekly content refresh priority ranking
Action                : Editorial audit, content expansion, meta/title updates
Evaluation Metric     : Precision@50 & Top-K Review Efficiency
Primary Risk          : Wasted editorial audit hours on false positives


## 3. Quick look at the data (2-3 real numbers)

Below we load the starter dataset (`data/raw/content_refresh_anonymized.csv`) and compute three empirical data points that motivate this lane choice.

In [3]:
import pandas as pd
import numpy as np
import json

# Load starter dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
print(f'Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns across {df["client_id"].nunique()} clients\n')

# Number 1: Decline prevalence across dataset
df['is_declining'] = df['trend_direction'].str.lower().eq('down')
declining_count = df['is_declining'].sum()
declining_pct = df['is_declining'].mean() * 100
print(f'1. Overall Decline Rate: {declining_count:,} / {len(df):,} pages ({declining_pct:.1f}%) are currently in a downward trend.')

# Number 2: High-Demand Stale Content Segment
stale_visible = df[(df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500)]
print(f'2. High-Demand Stale Pages: {len(stale_visible):,} pages ({len(stale_visible)/len(df)*100:.2f}%) are stale (>=180d since update) AND have high demand (>=500 impressions).')
print(f'   - Average impressions for this segment: {stale_visible["impressions_90d"].mean():,.0f}')

# Number 3: Empirical Lift of Learned ML over Rule Baseline
try:
    res = json.load(open('outputs/model_results.json'))
    base_p50 = res['baseline']['baseline_precision_at_50']
    rf_p50 = res['models']['random_forest']['precision_at_50']
    lift = rf_p50 / base_p50
    print(f'3. Model vs Baseline Lift: Baseline Precision@50 = {base_p50:.3f} (~{round(base_p50*50)}/50 right) vs Random Forest Precision@50 = {rf_p50:.3f} (~{round(rf_p50*50)}/50 right).')
    print(f'   - The learned model provides a {lift:.2f}x lift over a static hand-written rule.')
except Exception as e:
    print('3. Pipeline model results available after running scripts/run_all.py.')


Dataset Shape: 30,000 rows x 44 columns across 32 clients

1. Overall Decline Rate: 16,262 / 30,000 pages (54.2%) are currently in a downward trend.
2. High-Demand Stale Pages: 17 pages (0.06%) are stale (>=180d since update) AND have high demand (>=500 impressions).
   - Average impressions for this segment: 11,601
3. Model vs Baseline Lift: Baseline Precision@50 = 0.240 (~12/50 right) vs Random Forest Precision@50 = 0.740 (~37/50 right).
   - The learned model provides a 3.08x lift over a static hand-written rule.


## 4. Careful words: what I can and can't claim

To maintain strict scientific and professional rigor, all claims in this research track are bounded by the following rules:

### What I CAN claim:
- **Observed & Directional Relationships**: We observe empirical correlations between content attributes (age, position tier, CTR, scroll rate) and traffic decline patterns in the historical dataset.
- **Decision Support & Prioritization**: The model generates an evidence-backed priority score to help editorial teams allocate human review capacity to candidate pages with higher relative decline risk.
- **Holdout Validation Performance**: The model outperforms hand-written baseline rules on out-of-sample client holdout splits under the defined `is_declining_label` metric.

### What I CANNOT claim:
- **No Causal Proof**: We cannot claim that updating a recommended page *causes* a traffic recovery or ranking increase (observational data, no A/B testing or causal intervention design).
- **Not 'Predicting Google'**: We are not reverse-engineering or predicting Google's search algorithm; we are predicting internal historical performance patterns.
- **No Absolute Guarantees**: A high priority score indicates high historical signal alignment, not a guaranteed editorial return.

In [4]:
# 4. Claim Boundaries Assertion Check
allowed_claims = ['observed', 'directional', 'decision-support', 'holdout-validated']
disallowed_claims = ['causal proof', 'predicting Google', 'guaranteed traffic gain']

print('Claim Guidelines Standardized:')
print('ALLOWED terminology   :', ', '.join(allowed_claims))
print('DISALLOWED terminology:', ', '.join(disallowed_claims))


Claim Guidelines Standardized:
ALLOWED terminology   : observed, directional, decision-support, holdout-validated
DISALLOWED terminology: causal proof, predicting Google, guaranteed traffic gain


## Self-check

Before submitting, verify all checks:

- [x] Every section above is filled — markdown thinking AND code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to repo under `work/notebooks/` and ready for portal submission.